In [2]:
# Data handling libraries
import json
import numpy as np
import pandas as pd
from pandas import json_normalize

# Natural Language Processing (NLP) libraries
from nltk.corpus import stopwords

# Scikit-learn modeling libraries
from sklearn.dummy import DummyClassifier # For baseline model
from sklearn.feature_extraction.text import TfidfVectorizer # To convert text to numbers
from sklearn.linear_model import LogisticRegression # The classifier model
from sklearn.metrics import accuracy_score, classification_report # For evaluation
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score # For splitting and validating
from sklearn.pipeline import Pipeline # To chain processing steps

In [3]:
from datasets import Dataset
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.utils.class_weight import compute_class_weight

import evaluate
import torch
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          DataCollatorWithPadding, TrainingArguments, Trainer,
                          EarlyStoppingCallback)

# 1. Data Loading

In [4]:
# Load the training data from a JSON Lines file (one JSON object per line)
train_data = pd.read_json(rf'..\\Data\\train.jsonl', lines=True)
# The tweet data is nested. json_normalize flattens the nested JSON into columns.
train_data = json_normalize(train_data.to_dict(orient='records'))

# Load the Kaggle test data (which we will make predictions on)
kaggle_data = pd.read_json(rf'..\\Data\\kaggle_test.jsonl', lines=True)
# Also normalize the Kaggle data
kaggle_data = json_normalize(kaggle_data.to_dict(orient='records'))

False
no gpu


In [9]:
import socket, os, platform, sys
print("HOSTNAME:", socket.gethostname())
print("USER:", os.getenv("USER") or os.getenv("USERNAME"))
print("OS:", platform.platform())
print("PYTHON:", sys.executable)
print("CWD:", os.getcwd())

HOSTNAME: HPSpectrePablo
USER: poule
OS: Windows-11-10.0.26100-SP0
PYTHON: c:\Users\poule\anaconda3\envs\inf554\python.exe
CWD: c:\Users\poule\Documents\TAFF\3A\DEEP_LEARNING\projet\DLTweets


In [4]:
# Define a function to get the full text from a tweet object.
# Tweets can be truncated, storing the full version in 'extended_tweet.full_text'.
def extract_full_text(tweet):
    # Start with the standard 'text' field
    text = tweet['text']
    # Check if the 'extended_tweet.full_text' field exists (is not NaN)
    if not pd.isna(tweet['extended_tweet.full_text']):
        # If it exists, it's the full text, so use it instead
        text = tweet['extended_tweet.full_text']
    return text

# Apply this function to every row (axis=1) in the training data
train_data['full_text'] = train_data.apply(lambda tweet: extract_full_text(tweet), axis=1)
# Apply the same function to the test data
kaggle_data['full_text'] = kaggle_data.apply(lambda tweet: extract_full_text(tweet), axis=1)

In [7]:
SEED = 10
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

MODEL_NAME = "vinai/bertweet-base"   # ou "roberta-base"
TEXT_COL   = "full_text"
LABEL_COL  = "label"
ID_COL     = "id_str"              # pour la soumission
MAX_LEN    = 100

## ON TESTE SUR UN PETIT VOLUME DE DONNEES :

In [1]:
#### on teste sur petit volume de données : 

FRAC = 0.001          # 0.1%
LABEL_COL = "label"  # adapte si besoin

def stratified_fraction(df, label_col=LABEL_COL, frac=FRAC, min_per_class=1, seed=42):
    out = []
    for y, g in df.groupby(label_col):
        n = max(min_per_class, int(round(len(g) * frac)))
        n = min(n, len(g))  # sécurité
        out.append(g.sample(n=n, random_state=seed))
    return pd.concat(out).sample(frac=1.0, random_state=seed).reset_index(drop=True)

# Exemple d'usage sur ton train
train_data = stratified_fraction(train_data, LABEL_COL, FRAC)
print(train_data.shape, train_data[LABEL_COL].value_counts(normalize=True))


NameError: name 'train_data' is not defined

## SPLIT TRAIN/VAL

In [8]:

sss = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=SEED)
tr_idx, va_idx = next(sss.split(train_data, train_data[LABEL_COL]))
train_data, val_data = train_data.iloc[tr_idx].reset_index(drop=True), train_data.iloc[va_idx].reset_index(drop=True)

print("Split:", train_data.shape, val_data.shape, " | Ratio labels (train):",
      train_data[LABEL_COL].mean(), " | (val):", val_data[LABEL_COL].mean())

Split: (124, 194) (31, 194)  | Ratio labels (train): 0.46774193548387094  | (val): 0.45161290322580644


## CONSTRUCTION DES DS POUR TORCH :

In [9]:

# =======================
# 0) Configs / utilitaires
# =======================

def normalize_tweet(text: str) -> str:
    """Petit nettoyage non destructif: conserve le signal (mentions, hashtags, emojis)."""
    if not isinstance(text, str):
        return ""
    # simplifier les URLs -> 'http'
    t = text.replace("https://", "http ").replace("http://", "http ")
    return t


def preprocess(batch):
    texts = batch["full_text"]  # adapte le nom de ta colonne texte
    return tokenizer(
        texts,
        truncation=True,
        max_length=MAX_LEN,
        padding="max_length"  # on laisse le collator gérer le padding dynamique
    )

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)

def preprocess(batch):
    texts = [normalize_tweet(t) for t in batch[TEXT_COL]]
    return tokenizer(
        texts,
        truncation=True,
        max_length=MAX_LEN,
        padding="max_length"
    )
# 1) Construire un jeu "propre" (enlever TOUT le reste)
cols_keep = ["full_text", "label"]  # pas d'autres colonnes !
ds_train = Dataset.from_pandas(train_data[cols_keep].reset_index(drop=True))
ds_val = Dataset.from_pandas(val_data[cols_keep].reset_index(drop=True))
ds_kaggle = Dataset.from_pandas(kaggle_data[["full_text"]].reset_index(drop=True))

# 2) Tokeniser
ds_train = ds_train.map(preprocess, batched=True, remove_columns=ds_train.column_names)  # <- retire les anciennes colonnes
ds_val = ds_val.map(preprocess, batched=True, remove_columns=ds_val.column_names)
ds_kaggle = ds_kaggle.map(preprocess, batched=True, remove_columns=ds_kaggle.column_names)

# 3) (ré)ajouter les labels après map si remove_columns les a enlevés
ds_train = ds_train.add_column("labels", train_data["label"].astype(int).tolist())
ds_val = ds_val.add_column("labels", val_data["label"].astype(int).tolist())
ds_kaggle = ds_kaggle  # pas de labels dans le jeu de test Kaggle

# 4) Formater pour torch avec UNIQUEMENT ces colonnes
model_input_cols = ["input_ids", "attention_mask", "labels"]
ds_train = ds_train.with_format("torch", columns=model_input_cols)
ds_val = ds_val.with_format("torch", columns=model_input_cols)
ds_kaggle = ds_kaggle.with_format("torch", columns=["input_ids", "attention_mask"])

# 5) Collator qui padde de façon cohérente toutes les clés d'inputs
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)


Map:   0%|          | 0/124 [00:00<?, ? examples/s]

Map:   0%|          | 0/31 [00:00<?, ? examples/s]

Map:   0%|          | 0/103380 [00:00<?, ? examples/s]

In [11]:
# =======================
# 4) Modèle + class weights
# =======================
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)

# Poids de classes (optionnel, utile si léger déséquilibre)
class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.array([0,1]),
    y=train_data[LABEL_COL].values
)
class_weights = torch.tensor(class_weights, dtype=torch.float)

def compute_weighted_loss(model, inputs, return_outputs=False,**kwargs):
    labels = inputs.pop("labels")
    outputs = model(**inputs)
    logits = outputs.logits
    loss_fct = torch.nn.CrossEntropyLoss(weight=class_weights.to(logits.device))
    loss = loss_fct(logits, labels)
    return (loss, outputs) if return_outputs else loss

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at vinai/bertweet-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [12]:
# =======================
# 5) Entraînement
# =======================
acc_metric = evaluate.load("accuracy")
f1_metric  = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = logits.argmax(axis=-1)
    return {
        "accuracy": acc_metric.compute(predictions=preds, references=labels)["accuracy"],
        "f1": f1_metric.compute(predictions=preds, references=labels, average="weighted")["f1"]
    }

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

training_args = TrainingArguments(
    output_dir="./checkpoints_ft",
    eval_strategy="steps",
    eval_steps=200,
    save_strategy="steps",
    save_steps=200,
    save_total_limit=2,
    metric_for_best_model="accuracy",
    load_best_model_at_end=True,
    logging_steps=50,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=4,
    learning_rate=2e-5,
    weight_decay=0.01,
    warmup_ratio=0.1,
    lr_scheduler_type="linear",
    seed=SEED,
    bf16=torch.cuda.is_available(),   # True si GPU BF16; sinon False
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=ds_train,
    eval_dataset=ds_val,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
)
# injecter la loss pondérée (sinon commente la ligne ci-dessous pour loss standard)
trainer.compute_loss = compute_weighted_loss

trainer.train()
print(trainer.evaluate())


C:\Users\poule\AppData\Local\Temp\ipykernel_21940\1966739641.py:38: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
C:\Users\poule\AppData\Roaming\Python\Python312\site-packages\torch\utils\data\dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Step,Training Loss,Validation Loss


C:\Users\poule\AppData\Roaming\Python\Python312\site-packages\torch\utils\data\dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


{'eval_loss': 0.7200148701667786, 'eval_accuracy': 0.3548387096774194, 'eval_f1': 0.3548387096774194, 'eval_runtime': 3.5948, 'eval_samples_per_second': 8.624, 'eval_steps_per_second': 0.278, 'epoch': 4.0}
